# Instruction-tune AxiomicLabs/GPT-X2.5-135M on an Input/Output JSONL

Self-contained except for the dataset: it downloads the model from the Hugging
Face Hub (custom `GPTX2` code, `trust_remote_code=True`), applies the model's
native **ChatML** tokens (`<|im_start|>` / `<|im_end|>`), and does masked-loss
SFT — only completion tokens are supervised, prompts are masked out.

**Dataset format** (`data/instruct.jsonl`, one JSON object per line):

```json
{"input": "What is the capital of France?", "output": "The capital of France is Paris."}
```

Rows with `instruction` / `output` keys also work (alpaca-style); anything else
fails loudly with the keys it found.

Set `SMOKE = True` for a ~2-minute CPU sanity run (tiny subset, few steps). The
defaults assume a single 16GB GPU (~30–60 min for a few thousand examples).

In [ ]:
# @title 0 · Config
SMOKE = True  # @param {type:"boolean"}
DATA_PATH = "data/instruct.jsonl"  # @param {type:"string"}
OUTPUT_DIR = "runs/gptx25-135m-sft"  # @param {type:"string"}

MODEL_ID = "AxiomicLabs/GPT-X2.5-135M"
LEARNING_RATE = 1e-5
EPOCHS = 3
BATCH_SIZE = 2  # per-device micro batch
GRAD_ACCUM = 8  # effective batch = 16
MAX_LEN = 1024
WARMUP_RATIO = 0.03
LOG_EVERY = 10
SAVE_EVERY_STEPS = 200
SEED = 1337

# Only used when SMOKE=True: keeps the test path tiny and offline-friendly.
SMOKE_MAX_EXAMPLES = 12
SMOKE_STEPS = 4

import json
import math
import os
import random
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset

assert torch.cuda.is_available() or SMOKE, (
    "no CUDA device; set SMOKE=True for the CPU sanity run"
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def set_seed(seed: int) -> None:
    """Seed everything that influences data order and init."""
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE}  dtype={AMP_DTYPE}  smoke={SMOKE}")

In [ ]:
# @title 1 · Load model + tokenizer (custom GPTX2 code from the Hub)
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# ChatML ids ship with the tokenizer (32,770-vocab = 32K BPE + ChatML pair).
IM_START = tok.convert_tokens_to_ids("<|im_start|>")
IM_END = tok.convert_tokens_to_ids("<|im_end|>")
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else IM_END
assert IM_START is not None and IM_END is not None and IM_START != IM_END, (
    "ChatML tokens missing from tokenizer; the notebook needs the stock 32,770-token set"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32,
    attn_implementation="eager",  # GPTX2 uses XSA projection; eager is the tested path
)
model.to(DEVICE)
model.gradient_checkpointing_disable()
model.config.use_cache = False  # incompatible with grad checkpointing-free SFT batches
print(model.num_parameters(), "parameters")
print("ChatML:", IM_START, IM_END, "pad:", PAD_ID)

In [ ]:
# @title 2 · Load the Input/Output JSONL
def load_io_jsonl(path: str | Path, limit: int | None = None) -> list[dict[str, str]]:
    """Read {input, output} (or {instruction, output}) rows; fail loudly on junk."""
    rows: list[dict[str, str]] = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path}:{line_no} is not valid JSON: {exc}") from exc
            prompt = record.get("input", record.get("instruction"))
            completion = record.get("output", record.get("response"))
            if not prompt or not completion:
                raise ValueError(
                    f"{path}:{line_no} needs 'input'+'output' (or 'instruction'+'output'); "
                    f"found keys {sorted(record)}"
                )
            rows.append({"input": str(prompt), "output": str(completion)})
            if limit is not None and len(rows) >= limit:
                break
    if not rows:
        raise ValueError(f"{path} contains no usable rows")
    return rows


if SMOKE and not Path(DATA_PATH).exists():
    # Sanity fixture so the notebook runs end-to-end before real data exists.
    Path(DATA_PATH).parent.mkdir(parents=True, exist_ok=True)
    fixture = [
        {"input": "What is 2 plus 2?", "output": "2 plus 2 is 4."},
        {"input": "What is the capital of France?", "output": "The capital of France is Paris."},
        {"input": "Say hello in one word.", "output": "Hello."},
        {"input": "What colour is the sky on a clear day?", "output": "The sky is blue."},
    ]
    with Path(DATA_PATH).open("w", encoding="utf-8") as handle:
        for row in fixture:
            handle.write(json.dumps(row) + "\n")
    print(f"wrote smoke fixture -> {DATA_PATH}")

rows = load_io_jsonl(DATA_PATH, limit=SMOKE_MAX_EXAMPLES if SMOKE else None)
print(f"{len(rows)} examples from {DATA_PATH}; first: {rows[0]['input'][:60]!r}")

In [ ]:
# @title 3 · Render ChatML with completion-only labels
IGNORE_INDEX = -100  # CE loss ignores these positions


def render_example(row: dict[str, str]) -> dict[str, list[int]]:
    """<|im_start|>user\n{input}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>

    Labels supervise only the assistant completion and its <|im_end|>.
    """
    prompt_text = f"<|im_start|>user\n{row['input']}<|im_end|>\n<|im_start|>assistant\n"
    completion_text = f"{row['output']}<|im_end|>"
    prompt_ids = tok.encode(prompt_text, add_special_tokens=False)
    completion_ids = tok.encode(completion_text, add_special_tokens=False)
    input_ids = (prompt_ids + completion_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + list(completion_ids))[:MAX_LEN]
    if all(label == -100 for label in labels):
        raise ValueError(f"prompt fills MAX_LEN={MAX_LEN}; raise it or shorten the input")
    return {"input_ids": input_ids, "labels": labels}


class SftDataset(Dataset):
    """Right-padded batch of rendered examples."""

    def __init__(self, examples: list[dict[str, str]]):
        self.items = [render_example(row) for row in examples]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        return self.items[index]


def collate(batch: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    width = max(len(item["input_ids"]) for item in batch)
    input_ids, labels, attention = [], [], []
    for item in batch:
        pad = width - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [PAD_ID] * pad)
        labels.append(item["labels"] + [-100] * pad)
        attention.append([1] * len(item["input_ids"]) + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention, dtype=torch.long),
    }


split = max(1, int(0.95 * len(rows)))
train_rows = rows[:split]
val_rows = rows[split:] or rows[-1:]
train_ds = SftDataset(train_rows)
val_ds = SftDataset(val_rows)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate)
probe = collate([train_ds[0]])
print("supervised tokens in example 0:", int((probe['labels'][0] != -100).sum()))
print(tok.decode([i for i, l in zip(probe['input_ids'][0], probe['labels'][0]) if l != -100]))

In [ ]:
# @title 4 · Optimizer, schedule, eval
import torch.nn.functional as F

decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    (no_decay if (param.ndim < 2 or "norm" in name.lower()) else decay).append(param)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.01}, {"params": no_decay, "weight_decay": 0.0}],
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
)

steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
if SMOKE:
    total_steps = SMOKE_STEPS
else:
    total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

def lr_at(step: int) -> float:
    """Cosine decay with linear warmup."""
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)
scaler = torch.amp.GradScaler(enabled=AMP_DTYPE == torch.float16)

@torch.no_grad()
def evaluate(max_batches: int = 20) -> float:
    """Mean supervised-token loss on the validation rows."""
    model.eval()
    total, seen = 0.0, 0
    for index, batch in enumerate(val_loader):
        if index >= max_batches:
            break
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        shift_logits = out.logits[..., :-1, :].float()
        shift_labels = batch["labels"][..., 1:]
        loss = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=-100,
        )
        if torch.isfinite(loss):
            total += float(loss)
            seen += 1
    model.train()
    return total / max(1, seen)

print(f"total_steps={total_steps}  steps/epoch={steps_per_epoch}  warmup={warmup_steps}")

In [ ]:
# @title 5 · Train
model.train()
step = 0
micro = 0
accumulated = 0.0
losses: list[float] = []
t0 = time.perf_counter()
done = False
for epoch in range(EPOCHS if not SMOKE else 100):  # SMOKE stops by step count
    if done:
        break
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            out = model(**batch)
            shift_logits = out.logits[..., :-1, :].float()
            shift_labels = batch["labels"][..., 1:]
            loss = F.cross_entropy(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1),
                ignore_index=-100,
            ) / GRAD_ACCUM
        scaler.scale(loss).backward()
        accumulated += float(loss.detach())
        micro += 1
        if micro % GRAD_ACCUM:
            continue
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1
        losses.append(accumulated)
        if step % LOG_EVERY == 0 or step == 1:
            recent = sum(losses[-LOG_EVERY:]) / len(losses[-LOG_EVERY:])
            rate = step / max(1e-6, time.perf_counter() - t0)
            print(f"step {step:>5}/{total_steps}  loss {recent:.4f}  lr {scheduler.get_last_lr()[0]:.2e}  {rate:.2f} it/s", flush=True)
        if step % SAVE_EVERY_STEPS == 0:
            model.save_pretrained(Path(OUTPUT_DIR) / "checkpoint")
            tok.save_pretrained(Path(OUTPUT_DIR) / "checkpoint")
        if step >= total_steps:
            done = True
            break

val_loss = evaluate()
print(f"done: {step} steps, train loss {sum(losses[-LOG_EVERY:]) / max(1, len(losses[-LOG_EVERY:])):.4f}, val loss {val_loss:.4f}")

In [ ]:
# @title 6 · Save + smoke-generate
model.config.use_cache = True
final_dir = Path(OUTPUT_DIR) / "final"
model.save_pretrained(final_dir)
tok.save_pretrained(final_dir)
print(f"saved -> {final_dir}")

model.eval()
prompts = [row["input"] for row in val_rows[:3]]
for prompt in prompts:
    text = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    ids = tok.encode(text, add_special_tokens=False, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=48 if not SMOKE else 12,
            do_sample=not SMOKE,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=IM_END,
            pad_token_id=PAD_ID,
        )
    completion = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    print(f">> {prompt!r}\n   {completion.strip()!r}")

summary = {
    "model": MODEL_ID,
    "data": str(DATA_PATH),
    "examples": len(rows),
    "steps": step,
    "train_loss": round(sum(losses[-LOG_EVERY:]) / max(1, len(losses[-LOG_EVERY:])), 4) if losses else None,
    "val_loss": round(val_loss, 4) if val_loss == val_loss else None,
    "smoke": SMOKE,
    "output": str(final_dir),
}
(Path(OUTPUT_DIR) / "run_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))